# McDonald's Food Demand Prediction — Data Preprocessing

This notebook loads the **3,035-row synthetic raw dataset**, identifies data-quality problems, cleans the data, and prepares the five main features for machine learning.

### Main prediction features
- `Previous_Day_Sales`
- `Customers`
- `Promotion`
- `Is_Weekend`
- `Food_Category`

### Target
- `Sales`

> This is a synthetic academic dataset and does not contain McDonald's real internal data.


## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")


## 2. Load the Raw Dataset

Upload `mcdonalds_food_demand_raw_3000.csv` to Colab, then run the cell below.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


## 3. Initial Data Inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
df.describe(include="all").T


## 4. Check Data Quality Problems

The raw dataset intentionally contains duplicates, missing values, inconsistent categorical values, invalid numeric values, and a few sales outliers.

In [ ]:
# Check categorical values before cleaning

print("Weather values:")
print(df["Weather"].value_counts(dropna=False))

print("\nFood Category values:")
print(df["Food_Category"].value_counts(dropna=False))

print("\nInvalid Customers:")
print((df["Customers"] < 0).sum())

print("\nInvalid Prices:")
print((df["Average_Price"] < 0).sum())

print("\nPotential Sales Outliers:")
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Outliers:", ((df["Sales"] < lower) | (df["Sales"] > upper)).sum())


## 5. Remove Exact Duplicate Records

In [ ]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicate rows remaining:", df.duplicated().sum())


## 6. Standardize Categorical Values

Convert inconsistent spellings, spaces, and capitalization into consistent categories.

In [ ]:
# Clean Weather
df["Weather"] = (
    df["Weather"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Clean Food Category
df["Food_Category"] = (
    df["Food_Category"]
    .astype("string")
    .str.strip()
    .str.title()
)

print("Cleaned Weather values:")
print(df["Weather"].value_counts(dropna=False))

print("\nCleaned Food Category values:")
print(df["Food_Category"].value_counts(dropna=False))


## 7. Convert Data Types

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_columns = [
    "Is_Holiday",
    "Is_Weekend",
    "Promotion",
    "Average_Price",
    "Previous_Day_Sales",
    "Customers",
    "Sales"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)


## 8. Handle Invalid Numeric Values

Negative customers and negative prices are not meaningful for this project, so they are converted to missing values before imputation.

In [ ]:
df.loc[df["Customers"] < 0, "Customers"] = np.nan
df.loc[df["Average_Price"] < 0, "Average_Price"] = np.nan
df.loc[df["Previous_Day_Sales"] < 0, "Previous_Day_Sales"] = np.nan
df.loc[df["Sales"] < 0, "Sales"] = np.nan

print("Invalid values converted to NaN.")


## 9. Handle Missing Values

In [ ]:
print("Missing values before treatment:")
print(df.isnull().sum())


In [ ]:
# Fill categorical missing values with the mode
categorical_columns = ["Day_of_Week", "Food_Category", "Weather"]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill numerical feature missing values with the median
numerical_features = [
    "Average_Price",
    "Previous_Day_Sales",
    "Customers"
]

for col in numerical_features:
    df[col] = df[col].fillna(df[col].median())

# Sales is the target. Since only a small number of target values are missing,
# remove those rows rather than inventing target values.
df = df.dropna(subset=["Sales"]).reset_index(drop=True)

print("Missing values after treatment:")
print(df.isnull().sum())


## 10. Handle Sales Outliers

Use the IQR method to identify extreme sales values. For this prediction project, extreme target values are removed so they do not dominate model training.

In [ ]:
Q1 = df["Sales"].quantile(0.25)
Q3 = df["Sales"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (df["Sales"] < lower_bound) | (df["Sales"] > upper_bound)

print("Number of sales outliers:", outlier_mask.sum())

df = df.loc[~outlier_mask].reset_index(drop=True)

print("Shape after outlier treatment:", df.shape)


## 11. Feature Selection

For the final model, we use five main features that are easy to explain and directly related to demand.

In [ ]:
main_features = [
    "Previous_Day_Sales",
    "Customers",
    "Promotion",
    "Is_Weekend",
    "Food_Category"
]

target = "Sales"

model_df = df[main_features + [target]].copy()

print("Selected features:")
print(main_features)

print("\nTarget:")
print(target)

model_df.head()


## 12. Encode the Food Category

`Food_Category` is categorical, so we convert it into numerical values using `LabelEncoder`, consistent with the approach used in the original project.

In [ ]:
le_food = LabelEncoder()

model_df["Food_Category_enc"] = le_food.fit_transform(
    model_df["Food_Category"]
)

model_df.head()


## 13. Final ML Dataset

In [ ]:
X = model_df[
    [
        "Previous_Day_Sales",
        "Customers",
        "Promotion",
        "Is_Weekend",
        "Food_Category_enc"
    ]
]

y = model_df["Sales"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()


## 14. Final Data Quality Check

In [ ]:
print("Final dataset shape:", model_df.shape)
print("\nMissing values:")
print(model_df.isnull().sum())

print("\nDuplicate rows:")
print(model_df.duplicated().sum())

print("\nFinal feature columns:")
print(X.columns.tolist())


## 15. Save the Cleaned Dataset

The cleaned dataset can now be used for EDA, train-test splitting, Random Forest regression, and model evaluation.

In [ ]:
cleaned_output = "/content/mcdonalds_food_demand_cleaned.csv"

model_df.to_csv(cleaned_output, index=False)

print("Cleaned dataset saved to:", cleaned_output)

# Optional download
from google.colab import files
files.download(cleaned_output)


## Preprocessing Summary

The raw dataset was processed using:

1. Duplicate removal
2. Categorical value standardization
3. Date and numerical type conversion
4. Invalid-value detection and treatment
5. Missing-value imputation
6. Sales outlier detection using IQR
7. Feature selection
8. Label encoding of `Food_Category`

The final model-ready features are:

**Previous_Day_Sales, Customers, Promotion, Is_Weekend, Food_Category_enc**

with **Sales** as the prediction target.


# 16. Exploratory Sales Trend Analysis

Before building the model, we analyze sales trends over time.

We will create:
1. Average sales per day
2. Average sales per month


In [ ]:
# Prepare a time-series copy for analysis
trend_df = df.copy()
trend_df["Date"] = pd.to_datetime(trend_df["Date"], errors="coerce")

# Average sales per day
daily_sales = (
    trend_df.groupby("Date", as_index=False)["Sales"]
    .mean()
    .sort_values("Date")
)

plt.figure(figsize=(14, 5))
plt.plot(daily_sales["Date"], daily_sales["Sales"])
plt.xlabel("Date")
plt.ylabel("Average Sales")
plt.title("Average Sales per Day")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Average sales per month
trend_df["Month"] = trend_df["Date"].dt.to_period("M").astype(str)

monthly_sales = (
    trend_df.groupby("Month", as_index=False)["Sales"]
    .mean()
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales["Month"], monthly_sales["Sales"], marker="o")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.title("Average Sales per Month")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


# 17. Train-Test Split

We use the five selected features to predict `Sales`.

**Features:** Previous_Day_Sales, Customers, Promotion, Is_Weekend, Food_Category

**Target:** Sales


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# 18. Random Forest Regression

Random Forest Regressor is used because the target variable `Sales` is continuous.

The model combines many decision trees and averages their predictions to produce the final sales prediction.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=None,
    min_samples_split=2,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Random Forest model trained successfully.")


# 19. Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Performance")
print("-----------------")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")


# 20. Actual vs Predicted Sales

In [ ]:
comparison = pd.DataFrame({
    "Actual_Sales": y_test.values,
    "Predicted_Sales": np.round(y_pred, 2)
})

comparison.head(15)


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    comparison["Actual_Sales"].values,
    label="Actual Sales"
)

plt.plot(
    comparison["Predicted_Sales"].values,
    label="Predicted Sales"
)

plt.xlabel("Test Sample")
plt.ylabel("Sales")
plt.title("Actual vs Predicted Sales")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(y_test, y_pred, alpha=0.6)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales — Random Forest")
plt.tight_layout()
plt.show()


# 21. Confusion Matrix — Demand Category Evaluation

A confusion matrix is normally used for **classification**, while our main model is a regression model.

Therefore, we do **not** use a confusion matrix directly on continuous sales values.

Instead, we convert sales into three demand categories:

- Low Demand
- Medium Demand
- High Demand

We then compare the actual demand category with the category predicted by Random Forest.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Create demand categories using the actual-sales distribution
low_cutoff = y_train.quantile(0.33)
high_cutoff = y_train.quantile(0.67)

def demand_category(values):
    return pd.cut(
        values,
        bins=[-np.inf, low_cutoff, high_cutoff, np.inf],
        labels=["Low", "Medium", "High"]
    )

actual_category = demand_category(y_test)
predicted_category = demand_category(y_pred)

cm = confusion_matrix(
    actual_category,
    predicted_category,
    labels=["Low", "Medium", "High"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Low", "Medium", "High"]
)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax)
ax.set_title("Confusion Matrix — Actual vs Predicted Demand Category")
plt.tight_layout()
plt.show()


### Important note

The Random Forest model is still a **regression model** and its main evaluation metrics are **MAE, RMSE, and R²**.

The confusion matrix is an additional visualization that shows whether the model correctly identifies **low, medium, and high demand levels**.


# 22. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(
    feature_importance["Feature"],
    feature_importance["Importance"]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

feature_importance


# 23. Final Project Summary

### Prediction workflow

**Raw synthetic data → Data quality checks → Cleaning → Feature selection → Random Forest Regression → Sales prediction → Evaluation**

### Main features

1. Previous_Day_Sales
2. Customers
3. Promotion
4. Is_Weekend
5. Food_Category

### Target

**Sales**

### Evaluation

- MAE
- RMSE
- R²
- Actual vs Predicted Sales
- Demand-category Confusion Matrix

### Additional analysis

- Average Sales per Day
- Average Sales per Month
- Random Forest Feature Importance
